# Паттерн 3: Router / Triage — одноразовая маршрутизация

Самый простой мультиагентный паттерн: один узел классифицирует запрос, направляет его к нужному специалисту, тот отвечает — и граф завершается. Никаких циклов, никакого возврата к роутеру. Это «развилка на дороге» — одно решение и выход.

```mermaid
graph LR
    START((START)) --> router{{"🎯 Router<br/><i>классифицирует запрос</i>"}}
    router -->|billing| billing(["🔹 Billing<br/><i>обрабатывает запрос</i>"])
    router -->|technical| technical(["🔹 Technical<br/><i>обрабатывает запрос</i>"])
    router -->|general| general(["🔹 General<br/><i>обрабатывает запрос</i>"])
    billing --> END((END))
    technical --> END
    general --> END

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class router coordinator
    class billing,technical,general worker
    class START,END terminal
```

In [1]:
import sys
sys.path.insert(0, "..")

from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

llm = get_llm()

API key is set


## Состояние

Минимальное состояние: входной запрос, категория после классификации и финальный ответ. Reducer не нужен — каждое поле записывается ровно одним узлом. `recursion_limit` тоже не нужен — в графе нет циклов.

In [3]:
class RouterState(TypedDict):
    query: str
    category: str
    response: str

## Маршрутизатор

Роутер классифицирует запрос в одну из трёх категорий: `billing`, `technical`, `general`. LLM должна вернуть ровно одно слово. Если ответ не попадает ни в одну категорию — fallback на `general`, чтобы запрос не потерялся.

In [4]:
def router_node(state: RouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="""Ты — маршрутизатор службы поддержки.
Классифицируй запрос клиента. Ответь ОДНИМ словом:
- billing — вопросы по оплате, счетам, подписке, тарифам
- technical — технические проблемы, ошибки, настройки
- general — всё остальное: общие вопросы, пожелания, отзывы"""),
        HumanMessage(content=state["query"])
    ])
    category = response.content.strip().lower()
    if category not in ("billing", "technical", "general"):
        category = "general"  # fallback
    print(f"  [Роутер] Категория: {category}")
    return {"category": category}

## Специалисты

Каждый специалист получает сфокусированный системный промпт под свою область. После ответа — сразу в `END`, без возврата к роутеру. Один запрос — один специалист — один ответ.

In [5]:
def billing_agent(state: RouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — специалист по оплате. Ответь на вопрос клиента "
            "про счета, тарифы или подписку. Кратко: 2-3 предложения."
        )),
        HumanMessage(content=state["query"])
    ])
    print(f"  [Billing] Ответ готов")
    return {"response": response.content}


def technical_agent(state: RouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — техническая поддержка. Помоги клиенту решить "
            "техническую проблему. Кратко: 2-3 предложения."
        )),
        HumanMessage(content=state["query"])
    ])
    print(f"  [Technical] Ответ готов")
    return {"response": response.content}


def general_agent(state: RouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — оператор общей линии поддержки. Ответь на вопрос "
            "клиента дружелюбно и по делу. Кратко: 2-3 предложения."
        )),
        HumanMessage(content=state["query"])
    ])
    print(f"  [General] Ответ готов")
    return {"response": response.content}

## Сборка графа

Условное ребро из роутера ведёт к одному из трёх специалистов. Каждый специалист соединён напрямую с `END`. Итого два вызова LLM на запрос: один для маршрутизации, один для ответа. Это самый дешёвый мультиагентный паттерн.

In [6]:
def route(state: RouterState) -> str:
    return state["category"]


graph = StateGraph(RouterState)

graph.add_node("router", router_node)
graph.add_node("billing", billing_agent)
graph.add_node("technical", technical_agent)
graph.add_node("general", general_agent)

graph.add_edge(START, "router")
graph.add_conditional_edges("router", route, {
    "billing": "billing",
    "technical": "technical",
    "general": "general",
})
graph.add_edge("billing", END)
graph.add_edge("technical", END)
graph.add_edge("general", END)

app = graph.compile()

## Запуск

Проверим маршрутизацию на трёх запросах из разных категорий: технический, по оплате и общий отзыв. Каждый должен попасть к своему специалисту.

In [7]:
queries = [
    "Не могу зайти в аккаунт, пишет неверный пароль",
    "Сколько стоит подписка на год?",
    "Хочу оставить отзыв — ваш продукт отличный!",
]

for q in queries:
    print(f"\n  Запрос: \u00ab{q}\u00bb")
    result = app.invoke({"query": q, "category": "", "response": ""})
    print(f"  Категория: {result['category']}")
    print(f"  Ответ: {result['response'][:150]}...")


  Запрос: «Не могу зайти в аккаунт, пишет неверный пароль»


  [Роутер] Категория: technical


  [Technical] Ответ готов
  Категория: technical
  Ответ: Попробуйте сначала проверить раскладку клавиатуры, Caps Lock и убедиться, что пароль вводится без лишних пробелов. Если не поможет, нажмите «Забыли па...

  Запрос: «Сколько стоит подписка на год?»


  [Роутер] Категория: billing


  [Billing] Ответ готов
  Категория: billing
  Ответ: Годовая подписка обычно стоит как 12 месяцев по тарифу, но иногда действует скидка при оплате за год. Чтобы назвать точную цену, нужно знать, на какой...

  Запрос: «Хочу оставить отзыв — ваш продукт отличный!»


  [Роутер] Категория: general


  [General] Ответ готов
  Категория: general
  Ответ: Спасибо вам большое за тёплые слова — нам очень приятно! Передам ваш отзыв команде, нам важно знать, что продукт вам понравился....
